# TAPT 및 신규 단어집 적용에 따른 한국어 악성 채팅 분류 성능 비교

## 1. 평가 개요

동일한 테스트 데이터셋을 사용하여 다음 두 모델의 분류 성능을 비교한다.

- **기준 모델**: TAPT와 신규 단어집을 적용하지 않은 모델 (`normal_outputs`)
- **결합 적용 모델**: TAPT와 신규 단어집을 모두 적용한 모델 (`tapt_outputs`)
- **평가 데이터셋**
  - 정상 문장: `test_data/normal.csv`
  - 악성 문장: `test_data/toxic.csv`

평가 지표는 Accuracy, Macro F1, Toxic Precision, Toxic Recall, Toxic F1 및 혼동행렬이다. 모델별 전체 예측과 오분류 결과는 `comparison_results` 폴더에 저장한다.


## 2. 실행 환경

사용 가능한 장치를 CUDA, MPS, CPU 순서로 선택한다. MPS에서 지원되지 않는 연산이 발생하면 CPU에서 평가를 다시 수행한다.


## 3. 프로젝트 폴더 구성

노트북 파일을 프로젝트 폴더의 최상위에서 실행한다. 아래 폴더가 이미 있으면 그대로 사용하고, 필수 파일이 없으면 지정된 Google Drive ZIP을 자동으로 다운로드하여 압축을 해제한다.

```text
프로젝트폴더/
├─ compare_tapt.ipynb
├─ normal_outputs/
│  └─ final_model/
│     ├─ config.json
│     ├─ model.safetensors 또는 pytorch_model.bin
│     └─ tokenizer 관련 파일
├─ tapt_outputs/
│  └─ final_model/
│     ├─ config.json
│     ├─ model.safetensors 또는 pytorch_model.bin
│     └─ tokenizer 관련 파일
└─ test_data/
   ├─ normal.csv
   └─ toxic.csv
```

각 CSV에는 `text` 열이 있어야 한다. 최초 다운로드 시 인터넷 연결과 Google Drive 파일 접근 권한이 필요하다. 다운로드가 완료된 뒤에는 로컬 폴더를 재사용하므로 다시 다운로드하지 않는다.


## 4. 필수 라이브러리 설치

다음 셀은 현재 주피터 커널에 필요한 라이브러리가 없는 경우에만 설치한다.

설치가 끝난 뒤 import 오류가 발생하면 커널을 한 번 재시작하고 설치 셀 다음부터 다시 실행한다. CUDA 사용자는 운영체제와 CUDA 환경에 맞는 PyTorch가 별도로 필요할 수 있다.


In [ ]:
import importlib.util
import subprocess
import sys

# import 이름과 pip 패키지 이름이 다른 항목을 함께 관리한다.
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "torch": "torch",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "transformers": "transformers",
    "safetensors": "safetensors",
    "gdown": "gdown",
}

missing_packages = [
    package_name
    for import_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("설치할 라이브러리:", ", ".join(missing_packages))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages]
    )
    print("설치 완료. 이후 import 오류가 발생하면 커널을 재시작하세요.")
else:
    print("필수 라이브러리가 모두 설치되어 있습니다.")


## 5. 라이브러리 불러오기 및 경로 설정


In [ ]:
from pathlib import Path
import os
import gc
import re
import unicodedata
import platform
import shutil
import tempfile
import zipfile

# 일부 MPS 미지원 연산은 CPU에서 실행하도록 허용한다.
# torch를 import하기 전에 설정해야 한다.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import gdown

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# 노트북을 프로젝트 최상위 폴더에서 실행한다.
PROJECT_DIR = Path.cwd().resolve()

NORMAL_OUTPUT_DIR = PROJECT_DIR / "normal_outputs"
TAPT_OUTPUT_DIR = PROJECT_DIR / "tapt_outputs"

TEST_NORMAL_CSV = PROJECT_DIR / "test_data" / "normal.csv"
TEST_TOXIC_CSV = PROJECT_DIR / "test_data" / "toxic.csv"

RESULT_DIR = PROJECT_DIR / "comparison_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN = 64
MAX_CHARS = 64

LABEL2ID = {"normal": 0, "toxic": 1}
ID2LABEL = {0: "normal", 1: "toxic"}


def relative_path(path: Path) -> str:
    resolved = Path(path).resolve()
    try:
        return str(resolved.relative_to(PROJECT_DIR)) or "."
    except ValueError:
        return resolved.name


def select_device():
    """CUDA, Apple MPS, CPU 순서로 평가 장치를 선택한다."""
    if torch.cuda.is_available():
        return torch.device("cuda"), "NVIDIA CUDA"

    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend is not None and torch.backends.mps.is_available():
        return torch.device("mps"), "Apple Metal Performance Shaders(MPS)"

    return torch.device("cpu"), "CPU"


DEVICE, DEVICE_NAME = select_device()

# 장치별 기본 배치 크기
if DEVICE.type == "cuda":
    BATCH_SIZE = 128
elif DEVICE.type == "mps":
    BATCH_SIZE = 32
else:
    BATCH_SIZE = 16

print("=" * 72)
print("평가 환경 확인 결과")
print("=" * 72)
print(f"운영체제          : {platform.system()} {platform.release()}")
print(f"Python 버전       : {platform.python_version()}")
print(f"PyTorch 버전      : {torch.__version__}")
print(f"선택된 평가 장치  : {DEVICE_NAME} ({DEVICE})")
print(f"평가 배치 크기    : {BATCH_SIZE}")
print("프로젝트 경로     : .")
print(f"결과 저장 경로    : {relative_path(RESULT_DIR)}")

if DEVICE.type == "cuda":
    print(f"CUDA 장치명       : {torch.cuda.get_device_name(0)}")
elif DEVICE.type == "mps":
    print("Apple Silicon GPU 가속을 사용한다.")
else:
    print("GPU 가속을 사용할 수 없어 CPU로 평가한다.")


## 6. 누락된 모델 및 테스트 데이터 자동 다운로드

각 리소스의 필수 파일이 이미 있으면 다운로드를 건너뛴다. 폴더가 없거나 필수 파일이 빠져 있으면 해당 Google Drive ZIP을 다운로드한 뒤 프로젝트 폴더에 압축을 해제한다.

- `normal_outputs.zip`: 기준 모델 결과물
- `tapt_outputs.zip`: TAPT 및 신규 단어집 적용 모델 결과물
- `test_data.zip`: 평가용 `normal.csv`, `toxic.csv`


In [ ]:
RESOURCE_DOWNLOADS = {
    "normal_outputs": {
        "file_id": "1FM_NCsY5DOcoTEsBg5rWsMS8EMO2t-Zr",
        "target_dir": NORMAL_OUTPUT_DIR,
        "kind": "model",
    },
    "tapt_outputs": {
        "file_id": "1Mcl7DbRxRxDn09mTo3Q_P6mWcmFgZYus",
        "target_dir": TAPT_OUTPUT_DIR,
        "kind": "model",
    },
    "test_data": {
        "file_id": "16h6gh_dnzIZZZv8enpzAz_byKJ0jCtJR",
        "target_dir": TEST_NORMAL_CSV.parent,
        "kind": "test_data",
    },
}


def model_files_exist(output_dir: Path) -> bool:
    """출력 폴더 또는 final_model에 모델 설정과 가중치가 있는지 확인한다."""
    for candidate in (output_dir / "final_model", output_dir):
        config_exists = (candidate / "config.json").is_file()
        weight_exists = any(
            (candidate / filename).is_file()
            for filename in ("model.safetensors", "pytorch_model.bin")
        )
        if config_exists and weight_exists:
            return True
    return False


def test_files_exist(test_dir: Path) -> bool:
    """평가에 필요한 두 CSV가 모두 있는지 확인한다."""
    return all((test_dir / filename).is_file() for filename in ("normal.csv", "toxic.csv"))


def resource_is_ready(target_dir: Path, kind: str) -> bool:
    if kind == "model":
        return model_files_exist(target_dir)
    if kind == "test_data":
        return test_files_exist(target_dir)
    raise ValueError(f"지원하지 않는 리소스 종류입니다: {kind}")


def safe_extract_zip(zip_path: Path, extract_dir: Path) -> None:
    """ZIP 내부 경로가 압축 해제 폴더 밖으로 벗어나지 않도록 검사한 뒤 해제한다."""
    extract_root = extract_dir.resolve()

    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = (extract_dir / member.filename).resolve()
            if member_path != extract_root and extract_root not in member_path.parents:
                raise RuntimeError(f"안전하지 않은 ZIP 경로가 포함되어 있습니다: {member.filename}")
        archive.extractall(extract_dir)


def select_extracted_root(extract_dir: Path, expected_name: str) -> Path:
    """ZIP에 최상위 폴더가 포함된 경우와 내용만 포함된 경우를 모두 처리한다."""
    expected_root = extract_dir / expected_name
    if expected_root.is_dir():
        return expected_root

    entries = [
        path
        for path in extract_dir.iterdir()
        if path.name != "__MACOSX" and not path.name.startswith("._")
    ]
    if len(entries) == 1 and entries[0].is_dir():
        return entries[0]

    return extract_dir


def merge_directory(source_dir: Path, target_dir: Path) -> None:
    """압축 해제 결과를 대상 폴더에 병합한다."""
    target_dir.mkdir(parents=True, exist_ok=True)

    for source_item in source_dir.iterdir():
        if source_item.name == "__MACOSX" or source_item.name.startswith("._"):
            continue

        target_item = target_dir / source_item.name
        if source_item.is_dir():
            shutil.copytree(source_item, target_item, dirs_exist_ok=True)
        else:
            shutil.copy2(source_item, target_item)


def download_and_extract_resource(name: str, file_id: str, target_dir: Path) -> None:
    """Google Drive ZIP을 임시 폴더에 다운로드하고 대상 폴더로 병합한다."""
    print(f"[{name}] 필수 파일이 없어 다운로드를 시작합니다.")

    with tempfile.TemporaryDirectory(prefix=f".{name}_", dir=PROJECT_DIR) as temp_name:
        temp_dir = Path(temp_name)
        zip_path = temp_dir / f"{name}.zip"
        extract_dir = temp_dir / "extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)

        downloaded_path = gdown.download(
            id=file_id,
            output=str(zip_path),
            quiet=False,
        )
        if not downloaded_path or not zip_path.is_file():
            raise RuntimeError(f"[{name}] Google Drive 다운로드에 실패했습니다.")
        if not zipfile.is_zipfile(zip_path):
            raise RuntimeError(
                f"[{name}] 다운로드한 파일이 올바른 ZIP이 아닙니다. "
                "공유 권한 또는 파일 링크를 확인하세요."
            )

        safe_extract_zip(zip_path, extract_dir)
        source_dir = select_extracted_root(extract_dir, target_dir.name)
        merge_directory(source_dir, target_dir)

    print(f"[{name}] 준비 완료: {relative_path(target_dir)}")


def ensure_required_resources() -> None:
    for name, info in RESOURCE_DOWNLOADS.items():
        target_dir = info["target_dir"]
        kind = info["kind"]

        if resource_is_ready(target_dir, kind):
            print(f"[{name}] 기존 파일을 사용합니다: {relative_path(target_dir)}")
            continue

        download_and_extract_resource(
            name=name,
            file_id=info["file_id"],
            target_dir=target_dir,
        )

        if not resource_is_ready(target_dir, kind):
            raise RuntimeError(
                f"[{name}] 압축 해제 후에도 필수 파일을 찾지 못했습니다: "
                f"{relative_path(target_dir)}"
            )


ensure_required_resources()


## 7. 평가 대상 모델 및 데이터 파일 검증

학습 완료 모델은 각 출력 폴더의 `final_model`에 있어야 한다. 다음 셀은 모델 가중치, 설정 파일, 테스트 CSV의 존재 여부를 실행 전에 확인한다.


In [ ]:
def resolve_model_dir(output_dir: Path) -> Path:
    candidates = [
        output_dir / "final_model",
        output_dir,
    ]

    for candidate in candidates:
        config_exists = (candidate / "config.json").exists()
        model_exists = (
            (candidate / "model.safetensors").exists()
            or (candidate / "pytorch_model.bin").exists()
        )
        if config_exists and model_exists:
            return candidate

    raise FileNotFoundError(
        f"모델을 찾지 못했습니다: {relative_path(output_dir)}\n"
        "config.json과 model.safetensors 또는 pytorch_model.bin이 있는지 확인하세요."
    )


NORMAL_MODEL_DIR = resolve_model_dir(NORMAL_OUTPUT_DIR)
TAPT_MODEL_DIR = resolve_model_dir(TAPT_OUTPUT_DIR)

for file_path in [TEST_NORMAL_CSV, TEST_TOXIC_CSV]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"테스트 CSV를 찾지 못했습니다: {relative_path(file_path)}"
        )

print("기준 모델              :", relative_path(NORMAL_MODEL_DIR))
print("TAPT+신규 단어집 모델 :", relative_path(TAPT_MODEL_DIR))
print("normal 테스트          :", relative_path(TEST_NORMAL_CSV))
print("toxic 테스트           :", relative_path(TEST_TOXIC_CSV))


## 8. 테스트 데이터 로드 및 전처리

학습 코드와 동일한 기준으로 테스트 데이터를 전처리한다.

- NFKC 유니코드 정규화
- 제로폭 문자 제거
- 여러 공백을 한 칸으로 정리
- 빈 문장 제거
- 64자 초과 문장 제거
- 클래스 내부 중복 제거
- 동일 문장이 normal과 toxic 양쪽에 있으면 toxic 우선


In [ ]:
def preprocess_text(text) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"[\u200B-\u200D\u2060\uFEFF]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def read_test_csv(path: Path, label_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    if "text" not in df.columns:
        raise ValueError(
            f"{relative_path(path)}에 'text' 열이 없습니다. "
            f"현재 열: {list(df.columns)}"
        )

    before = len(df)

    df = df[["text"]].copy()
    df["text"] = df["text"].map(preprocess_text)
    df = df[df["text"] != ""]
    df = df[df["text"].str.len() <= MAX_CHARS]
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

    df["true_label"] = label_name
    df["true_id"] = LABEL2ID[label_name]

    print(f"{label_name}: 원본 {before:,}개 -> 평가 사용 {len(df):,}개")
    return df


normal_df = read_test_csv(TEST_NORMAL_CSV, "normal")
toxic_df = read_test_csv(TEST_TOXIC_CSV, "toxic")

# 양쪽 라벨에 같은 문장이 있으면 toxic 우선
overlap = set(normal_df["text"]) & set(toxic_df["text"])
if overlap:
    print(f"라벨 충돌 {len(overlap):,}개 발견: toxic 라벨을 우선합니다.")
    normal_df = normal_df[~normal_df["text"].isin(overlap)].reset_index(drop=True)

test_df = pd.concat([normal_df, toxic_df], ignore_index=True)

if test_df.empty:
    raise ValueError("평가할 테스트 문장이 없습니다.")

print()
print("최종 테스트 데이터 수:", len(test_df))
print(test_df["true_label"].value_counts())
display(test_df.head())

## 9. 모델 추론 및 성능 평가 함수 정의


In [ ]:
def _run_inference(model, tokenizer, data: pd.DataFrame, eval_device: torch.device):
    """지정된 장치에서 전체 테스트 문장에 대한 추론을 수행한다."""
    model.to(eval_device)
    model.eval()

    all_probs = []
    texts = data["text"].tolist()

    with torch.inference_mode():
        for start in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[start : start + BATCH_SIZE]

            encoded = tokenizer(
                batch_texts,
                max_length=MAX_LEN,
                truncation=True,
                padding=True,
                return_tensors="pt",
                return_token_type_ids=False,
            )
            encoded = {
                key: value.to(eval_device)
                for key, value in encoded.items()
            }

            output = model(**encoded, return_dict=True)
            probabilities = torch.softmax(output.logits, dim=-1)
            all_probs.append(probabilities.detach().cpu().numpy())

            completed = min(start + BATCH_SIZE, len(texts))
            print(
                f"  추론 진행률: {completed:,}/{len(texts):,}",
                end="\r",
            )

    print()
    return np.concatenate(all_probs, axis=0)


def predict_with_model(model_dir: Path, data: pd.DataFrame) -> pd.DataFrame:
    """모델을 평가하고, MPS 호환 오류 발생 시 CPU로 자동 재시도한다."""
    print(f"\n평가 모델 경로: {relative_path(model_dir)}")
    print(f"최초 평가 장치: {DEVICE}")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    eval_device = DEVICE

    try:
        probabilities = _run_inference(
            model=model,
            tokenizer=tokenizer,
            data=data,
            eval_device=eval_device,
        )
    except RuntimeError as error:
        if eval_device.type != "mps":
            raise

        print("\nMPS 연산 호환성 오류가 확인되어 CPU 평가를 재수행한다.")
        print(f"오류 요약: {str(error)[:300]}")

        del model
        gc.collect()
        if hasattr(torch, "mps"):
            torch.mps.empty_cache()

        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        eval_device = torch.device("cpu")
        probabilities = _run_inference(
            model=model,
            tokenizer=tokenizer,
            data=data,
            eval_device=eval_device,
        )

    pred_ids = probabilities.argmax(axis=1)

    result = data.copy()
    result["pred_id"] = pred_ids
    result["pred_label"] = result["pred_id"].map(ID2LABEL)
    result["normal_prob"] = probabilities[:, 0]
    result["toxic_prob"] = probabilities[:, 1]
    result["correct"] = result["true_id"] == result["pred_id"]
    result["evaluation_device"] = str(eval_device)

    del model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return result

def calculate_metrics(result_df: pd.DataFrame) -> dict:
    y_true = result_df["true_id"].to_numpy()
    y_pred = result_df["pred_id"].to_numpy()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "precision_toxic": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "recall_toxic": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "f1_toxic": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "normal_count": int((y_true == 0).sum()),
        "toxic_count": int((y_true == 1).sum()),
    }


def save_confusion_matrix(result_df: pd.DataFrame, model_name: str) -> None:
    y_true = result_df["true_id"].to_numpy()
    y_pred = result_df["pred_id"].to_numpy()

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    display_obj = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["normal", "toxic"],
    )

    fig, ax = plt.subplots(figsize=(5, 5))
    display_obj.plot(ax=ax, values_format="d")
    ax.set_title(f"{model_name} confusion matrix")
    fig.tight_layout()

    save_path = RESULT_DIR / f"{model_name}_confusion_matrix.png"
    fig.savefig(save_path, dpi=150)
    plt.show()

    print("혼동행렬 저장:", relative_path(save_path))

## 10. 모델별 성능 평가

두 모델을 순차적으로 불러와 동일한 테스트 데이터셋에서 평가한다. 성능 지표, 전체 예측, 오분류 결과 및 혼동행렬은 `comparison_results`에 저장한다.


In [ ]:
MODEL_PATHS = {
    "normal_model": NORMAL_MODEL_DIR,
    "tapt_new_words_model": TAPT_MODEL_DIR,
}

all_results = {}
metric_rows = []

for model_name, model_dir in MODEL_PATHS.items():
    print("=" * 70)
    print("평가 대상 모델:", model_name)

    result_df = predict_with_model(model_dir, test_df)
    metrics = calculate_metrics(result_df)

    all_results[model_name] = result_df
    metric_rows.append({"model": model_name, **metrics})

    prediction_path = RESULT_DIR / f"{model_name}_all_predictions.csv"
    wrong_path = RESULT_DIR / f"{model_name}_wrong_predictions.csv"

    result_df.to_csv(prediction_path, index=False, encoding="utf-8-sig")
    result_df[~result_df["correct"]].to_csv(
        wrong_path, index=False, encoding="utf-8-sig"
    )

    print(pd.Series(metrics))
    print("\n분류 보고서:")
    print(
        classification_report(
            result_df["true_id"],
            result_df["pred_id"],
            labels=[0, 1],
            target_names=["normal", "toxic"],
            digits=4,
            zero_division=0,
        )
    )

    save_confusion_matrix(result_df, model_name)

    print("전체 예측 저장:", relative_path(prediction_path))
    print("오분류 저장:", relative_path(wrong_path))

summary_df = pd.DataFrame(metric_rows).set_index("model")
summary_path = RESULT_DIR / "model_comparison_metrics.csv"
summary_df.to_csv(summary_path, encoding="utf-8-sig")

print("\n모델별 최종 성능 비교 결과")
display(summary_df)
print("성능 비교표 저장:", relative_path(summary_path))


## 11. 평가 지표 시각화

- `accuracy`: 전체 정답 비율
- `f1_macro`: normal과 toxic을 동일한 비중으로 계산한 평균 F1
- `precision_toxic`: toxic 예측 중 실제 toxic의 비율
- `recall_toxic`: 실제 toxic을 찾아낸 비율
- `f1_toxic`: toxic 정밀도와 재현율의 조화평균

클래스 분포가 불균형할 수 있으므로 Accuracy뿐 아니라 Macro F1과 Toxic F1도 함께 확인한다.


In [ ]:
plot_columns = ["accuracy", "f1_macro", "f1_toxic"]

ax = summary_df[plot_columns].plot(
    kind="bar",
    figsize=(9, 5),
    ylim=(0, 1),
    rot=0,
)
ax.set_title("Model performance comparison")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
ax.legend(loc="lower right")
plt.tight_layout()

chart_path = RESULT_DIR / "model_performance_comparison.png"
plt.savefig(chart_path, dpi=150)
plt.show()

print("그래프 저장:", relative_path(chart_path))

## 12. 모델 간 예측 차이 분석

두 모델의 예측 결과를 문장 단위로 비교하여 다음 사례를 구분한다.

- 기준 모델만 올바르게 분류한 사례
- 결합 적용 모델만 올바르게 분류한 사례
- 두 모델의 예측 결과가 서로 다른 사례


In [ ]:
normal_result = all_results["normal_model"].copy()
tapt_new_words_result = all_results["tapt_new_words_model"].copy()

comparison_df = normal_result[
    ["text", "true_label", "true_id"]
].copy()

comparison_df["normal_pred"] = normal_result["pred_label"]
comparison_df["normal_toxic_prob"] = normal_result["toxic_prob"]
comparison_df["normal_correct"] = normal_result["correct"]

comparison_df["tapt_new_words_pred"] = tapt_new_words_result["pred_label"]
comparison_df["tapt_new_words_toxic_prob"] = tapt_new_words_result["toxic_prob"]
comparison_df["tapt_new_words_correct"] = tapt_new_words_result["correct"]

comparison_df["prediction_changed"] = (
    comparison_df["normal_pred"] != comparison_df["tapt_new_words_pred"]
)

changed_df = comparison_df[comparison_df["prediction_changed"]].copy()
tapt_new_words_only_correct_df = comparison_df[
    (~comparison_df["normal_correct"])
    & (comparison_df["tapt_new_words_correct"])
].copy()
normal_only_correct_df = comparison_df[
    (comparison_df["normal_correct"])
    & (~comparison_df["tapt_new_words_correct"])
].copy()

comparison_df.to_csv(
    RESULT_DIR / "prediction_comparison_all.csv",
    index=False,
    encoding="utf-8-sig",
)
changed_df.to_csv(
    RESULT_DIR / "prediction_changed.csv",
    index=False,
    encoding="utf-8-sig",
)
tapt_new_words_only_correct_df.to_csv(
    RESULT_DIR / "tapt_new_words_only_correct.csv",
    index=False,
    encoding="utf-8-sig",
)
normal_only_correct_df.to_csv(
    RESULT_DIR / "normal_only_correct.csv",
    index=False,
    encoding="utf-8-sig",
)

print("모델 간 예측이 상이한 문장 수:", len(changed_df))
print(
    "TAPT+신규 단어집 모델만 올바르게 분류한 문장 수:",
    len(tapt_new_words_only_correct_df),
)
print("기준 모델만 올바르게 분류한 문장 수:", len(normal_only_correct_df))

display(changed_df.head(20))


## 13. 평가 결과 해석

- **Macro F1**은 normal과 toxic 클래스의 F1을 동일한 비중으로 평균한 지표이다.
- **Toxic F1**은 악성 문장 탐지에서 정밀도와 재현율의 균형을 나타낸다.
- **Toxic Recall**은 실제 악성 문장 중 모델이 toxic으로 탐지한 비율이다.
- 혼동행렬의 `normal → toxic`은 정상 문장 오탐, `toxic → normal`은 악성 문장 누락에 해당한다.
- 문장 단위 비교 결과는 두 모델의 개선 사례와 성능 저하 사례를 확인하는 데 사용한다.

## 14. 결과 요약

다음 셀은 기준 모델과 결합 적용 모델의 주요 지표 차이, 정상 문장 오탐 수, 악성 문장 누락 수를 정리한다.


In [ ]:
required_variables = {"summary_df", "all_results"}
missing_variables = required_variables - set(globals())

if missing_variables:
    raise RuntimeError("먼저 모델별 성능 평가 셀을 실행해야 합니다.")

baseline_name = "normal_model"
applied_name = "tapt_new_words_model"

metric_labels = {
    "accuracy": "Accuracy",
    "f1_macro": "Macro F1",
    "precision_toxic": "Toxic Precision",
    "recall_toxic": "Toxic Recall",
    "f1_toxic": "Toxic F1",
}

comparison_rows = []
for metric_key, metric_label in metric_labels.items():
    baseline_value = float(summary_df.loc[baseline_name, metric_key])
    applied_value = float(summary_df.loc[applied_name, metric_key])
    comparison_rows.append({
        "metric": metric_label,
        "baseline": baseline_value,
        "tapt_new_words": applied_value,
        "difference": applied_value - baseline_value,
    })

result_summary_df = pd.DataFrame(comparison_rows).set_index("metric")
display(result_summary_df.style.format("{:.4f}"))

baseline_cm = confusion_matrix(
    all_results[baseline_name]["true_id"],
    all_results[baseline_name]["pred_id"],
    labels=[0, 1],
)
applied_cm = confusion_matrix(
    all_results[applied_name]["true_id"],
    all_results[applied_name]["pred_id"],
    labels=[0, 1],
)

error_summary_df = pd.DataFrame(
    {
        "baseline": [int(baseline_cm[0, 1]), int(baseline_cm[1, 0])],
        "tapt_new_words": [int(applied_cm[0, 1]), int(applied_cm[1, 0])],
    },
    index=["정상 문장 오탐", "악성 문장 누락"],
)
error_summary_df["difference"] = (
    error_summary_df["tapt_new_words"] - error_summary_df["baseline"]
)
display(error_summary_df)

result_summary_path = RESULT_DIR / "model_comparison_differences.csv"
error_summary_path = RESULT_DIR / "classification_error_comparison.csv"
result_summary_df.to_csv(result_summary_path, encoding="utf-8-sig")
error_summary_df.to_csv(error_summary_path, encoding="utf-8-sig")

macro_difference = result_summary_df.loc["Macro F1", "difference"]
toxic_f1_difference = result_summary_df.loc["Toxic F1", "difference"]

print(f"Macro F1 변화: {macro_difference:+.4f}")
print(f"Toxic F1 변화: {toxic_f1_difference:+.4f}")
print(
    "정상 문장 오탐 변화: "
    f"{int(error_summary_df.loc['정상 문장 오탐', 'difference']):+d}건"
)
print(
    "악성 문장 누락 변화: "
    f"{int(error_summary_df.loc['악성 문장 누락', 'difference']):+d}건"
)
print("결과 차이는 TAPT와 신규 단어집을 함께 적용한 효과로 해석한다.")
print("지표 차이 저장:", relative_path(result_summary_path))
print("오류 비교 저장:", relative_path(error_summary_path))
